# Python File I/O

Small, runnable examples using `C:\data\samples`. Run the notebook from top to bottom. The setup creates a few harmless sample files used by later cells.

## 1. Setup sample files

In [ ]:
from pathlib import Path
import csv
import json

SAMPLES = Path(r"C:\data\samples")
SAMPLES.mkdir(parents=True, exist_ok=True)

(SAMPLES / "names.txt").write_text("Anu\nBala\nChen\nDivya\n", encoding="utf-8")
(SAMPLES / "numbers.txt").write_text("10\n20\nnot-a-number\n30\n", encoding="utf-8")
print(SAMPLES)

## 2. `open()` and explicit `close()`

A file object uses an operating-system resource. If you call `open()` yourself, close it when finished. `try/finally` guarantees cleanup even when reading fails.

In [ ]:
file = open(SAMPLES / "names.txt", mode="r", encoding="utf-8")
try:
    content = file.read()
    print(content)
    print("Closed inside try:", file.closed)
finally:
    file.close()

print("Closed after close():", file.closed)

## 3. Preferred style: the `with` statement

The context manager closes the file automatically when the block ends, including when an exception occurs.

In [ ]:
with open(SAMPLES / "names.txt", "r", encoding="utf-8") as file:
    first_five = file.read(5)
    print(first_five)
    print("Inside with:", file.closed)

print("Outside with:", file.closed)

## 4. Ways to read text

- `read()` returns all remaining text (or at most `size` characters).
- `readline()` returns one line.
- `readlines()` returns a list of lines.
- Iterating the file is memory-efficient and is preferred for large files.

In [ ]:
path = SAMPLES / "names.txt"

with path.open(encoding="utf-8") as file:
    print("read(3):", repr(file.read(3)))
    print("readline():", repr(file.readline()))
    print("remaining read():", repr(file.read()))

with path.open(encoding="utf-8") as file:
    print("readlines():", file.readlines())

In [ ]:
with (SAMPLES / "names.txt").open(encoding="utf-8") as file:
    for line_number, line in enumerate(file, start=1):
        print(line_number, line.rstrip("\n"))

## 5. Write, append, and exclusive creation

Common modes: `r` read, `w` overwrite/create, `a` append/create, and `x` create only if absent. Add `+` for reading and writing. Text mode is the default; binary mode uses `b`.

In [ ]:
output_path = SAMPLES / "greetings.txt"

with output_path.open("w", encoding="utf-8") as file:
    characters_written = file.write("Hello\n")
    file.writelines(["Good morning\n", "Welcome\n"])
print("write() returned:", characters_written)

with output_path.open("a", encoding="utf-8") as file:
    file.write("Appended line\n")

print(output_path.read_text(encoding="utf-8"))

In [ ]:
exclusive_path = SAMPLES / "created_once.txt"
try:
    with exclusive_path.open("x", encoding="utf-8") as file:
        file.write("This file was created with x mode.\n")
    print("Created:", exclusive_path)
except FileExistsError:
    print("Already exists; x mode did not overwrite it.")

## 6. `tell()`, `seek()`, and `truncate()`

`tell()` reports the current stream position. `seek(offset, whence)` moves it. In text files, reliably seek to positions previously returned by `tell()`; arbitrary offsets are safest in binary mode.

In [ ]:
with (SAMPLES / "names.txt").open("r", encoding="utf-8") as file:
    print("Start position:", file.tell())
    first_line = file.readline()
    saved_position = file.tell()
    second_line = file.readline()
    print(first_line.strip(), second_line.strip(), saved_position)

    file.seek(saved_position)
    print("Read second line again:", file.readline().strip())
    file.seek(0)
    print("Back at start:", file.readline().strip())

In [ ]:
truncate_path = SAMPLES / "truncate_demo.txt"
truncate_path.write_text("ABCDEFGHIJ", encoding="ascii")

with truncate_path.open("r+", encoding="ascii") as file:
    file.seek(5)
    file.truncate()

print(truncate_path.read_text(encoding="ascii"))  # ABCDE

## 7. Read and write in one open (`r+`, `w+`, `a+`)

`r+` requires an existing file and does not erase it. `w+` erases or creates. `a+` writes at the end; call `seek(0)` before reading from the beginning.

In [ ]:
update_path = SAMPLES / "update.txt"
update_path.write_text("cat dog", encoding="utf-8")

with update_path.open("r+", encoding="utf-8") as file:
    old_text = file.read()
    file.seek(0)
    file.write(old_text.replace("cat", "fox"))
    file.truncate()

print(update_path.read_text(encoding="utf-8"))

## 8. Binary files

Binary mode reads and writes `bytes`, not `str`. It is used for images, audio, archives, and other non-text data.

In [ ]:
binary_path = SAMPLES / "values.bin"

with binary_path.open("wb") as file:
    file.write(bytes([0, 1, 2, 127, 255]))

with binary_path.open("rb") as file:
    file.seek(-2, 2)  # two bytes before end; arbitrary relative seek is valid in binary mode
    print(file.read())

## 9. Encoding and newline handling

Specify an encoding for portable text files. `errors="replace"` can keep a read going when malformed bytes are acceptable; strict decoding is safer when bad data must be detected.

In [ ]:
unicode_path = SAMPLES / "unicode.txt"
with unicode_path.open("w", encoding="utf-8", newline="") as file:
    file.write("தமிழ்,हिन्दी,English\n")

with unicode_path.open("r", encoding="utf-8") as file:
    print(file.read())

## 10. CSV and JSON files

Use standard-library parsers instead of manually splitting structured data. Open CSV files with `newline=""`.

In [ ]:
csv_path = SAMPLES / "employees.csv"
rows = [{"id": 1, "name": "Anu"}, {"id": 2, "name": "Bala"}]

with csv_path.open("w", encoding="utf-8", newline="") as file:
    writer = csv.DictWriter(file, fieldnames=["id", "name"])
    writer.writeheader()
    writer.writerows(rows)

with csv_path.open(encoding="utf-8", newline="") as file:
    for row in csv.DictReader(file):
        print(row)

In [ ]:
json_path = SAMPLES / "config.json"
config = {"course": "Data Engineering", "active": True, "topics": ["files", "exceptions"]}

with json_path.open("w", encoding="utf-8") as file:
    json.dump(config, file, indent=2, ensure_ascii=False)

with json_path.open(encoding="utf-8") as file:
    loaded_config = json.load(file)
print(loaded_config)

## 11. File-system operations with `pathlib`

In [ ]:
for path in sorted(SAMPLES.iterdir()):
    if path.is_file():
        print(f"{path.name:20} {path.stat().st_size:4} bytes")

print("TXT files:", [path.name for path in SAMPLES.glob("*.txt")])

## 12. Exception basics: `try`, `except`, `else`, `finally`

Catch the narrow exception you can handle. `else` runs only when no exception occurs. `finally` always runs and is useful for cleanup; file cleanup is usually simpler with `with`.

In [ ]:
missing_path = SAMPLES / "missing.txt"

try:
    with missing_path.open(encoding="utf-8") as file:
        text = file.read()
except FileNotFoundError as error:
    print("Could not open:", error.filename)
else:
    print("Characters read:", len(text))
finally:
    print("Read attempt finished")

In [ ]:
try:
    value = int("abc")
except (TypeError, ValueError) as error:
    print(type(error).__name__, "-", error)

## 13. Handle bad records while iterating

Place `try` around the smallest operation that may fail so one bad line does not stop the whole file.

In [ ]:
valid_numbers = []
errors = []

with (SAMPLES / "numbers.txt").open(encoding="utf-8") as file:
    for line_number, line in enumerate(file, start=1):
        try:
            valid_numbers.append(int(line.strip()))
        except ValueError:
            errors.append((line_number, line.strip()))

print("Valid:", valid_numbers)
print("Rejected:", errors)

## 14. Raise and create exceptions

Use `raise` to reject invalid input. A custom exception gives callers a precise error type. `raise ... from error` preserves the original cause.

In [ ]:
class DataFileError(Exception):
    """Raised when a data file cannot be loaded correctly."""


def load_positive_integer(path):
    try:
        with Path(path).open(encoding="utf-8") as file:
            value = int(file.readline().strip())
    except (OSError, ValueError) as error:
        raise DataFileError(f"Cannot load an integer from {path}") from error

    if value <= 0:
        raise DataFileError("The integer must be positive")
    return value


print(load_positive_integer(SAMPLES / "numbers.txt"))

## 15. Re-raise after logging

In [ ]:
def read_required_text(path):
    try:
        return Path(path).read_text(encoding="utf-8")
    except OSError as error:
        print(f"Read failed: {error}")
        raise  # preserves the same exception and traceback


try:
    read_required_text(SAMPLES / "still_missing.txt")
except FileNotFoundError:
    print("Caller decided how to recover.")

## 16. Multiple files in one `with` statement

In [ ]:
source_path = SAMPLES / "names.txt"
copy_path = SAMPLES / "names_upper.txt"

with source_path.open(encoding="utf-8") as source, copy_path.open("w", encoding="utf-8") as target:
    for line in source:
        target.write(line.upper())

print(copy_path.read_text(encoding="utf-8"))

## Quick reference

| Mode | Meaning | Existing content | File must exist? |
|---|---|---|---|
| `r` | Read | Preserved | Yes |
| `w` | Write | Truncated | No |
| `a` | Append | Preserved; writes at end | No |
| `x` | Exclusive create | Never overwritten | No; fails if it exists |
| `r+` | Read/write | Preserved | Yes |
| `w+` | Read/write | Truncated | No |
| `a+` | Read/append | Preserved; writes at end | No |
| `rb`, `wb` | Binary read/write | Depends on base mode | Depends on base mode |

Best practices: use `with`, specify text encoding, iterate large files line by line, catch specific exceptions, and avoid catching bare `Exception` unless you will re-raise or handle it at an application boundary.

In [ ]:
assert (SAMPLES / "names.txt").exists()
assert file.closed
assert valid_numbers == [10, 20, 30]
assert errors == [(3, "not-a-number")]
assert loaded_config == config
assert (SAMPLES / "names_upper.txt").read_text(encoding="utf-8").startswith("ANU")
print("All File I/O and exception examples completed successfully.")